# Model Experiment: Prophet

Like SARIMA, Prophet is tested on the aggregated weekly series with minimal tuning. Prophet takes a different approach than SARIMA. instead of autoregression, it decomposes the series into additive components, trend, yearly seasonality, and holiday effects - each modeled separately and summed.

In [1]:
import mlflow
import dagshub
import numpy as np
import pandas as pd
from prophet import Prophet
import sys

sys.path.append("..")
from src.data.load_data import load_raw_data

dagshub.init(repo_owner="LukaBatilashvili07", repo_name="walmart-sales-forecasting", mlflow=True)
mlflow.set_experiment("Prophet_Training")

def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))

Importing plotly failed. Interactive plots will not work.


Accessing as LukaBatilashvili07

Initialized MLflow to track repo "LukaBatilashvili07/walmart-sales-forecasting"

Repository LukaBatilashvili07/walmart-sales-forecasting initialized!

2026/07/12 13:34:30 INFO mlflow.tracking.fluent: Experiment with name 'Prophet_Training' does not exist. Creating a new experiment.


## 1. Aggregate and Format for Prophet

Prophet requires columns named "ds" (date) and "y: (target). Holiday weeks are passed to Prophet's built-in "holidays" parameter, so it can model their effect explicitly instead of treating them as regular weeks.

In [2]:
train_raw, test_raw, stores, features = load_raw_data("../data/raw")

agg = train_raw.groupby("Date").agg(
    Weekly_Sales=("Weekly_Sales", "sum"),
    IsHoliday=("IsHoliday", "max"),
).reset_index().sort_values("Date")

n_valid = 39
train_agg = agg.iloc[:-n_valid]
valid_agg = agg.iloc[-n_valid:]

prophet_train = train_agg.rename(columns={"Date": "ds", "Weekly_Sales": "y"})[["ds", "y"]]

holiday_dates = agg.loc[agg["IsHoliday"], "Date"]
holidays_df = pd.DataFrame({
    "holiday": "walmart_holiday_week",
    "ds": holiday_dates,
    "lower_window": 0,
    "upper_window": 0,
})

print(f"Train: {len(prophet_train)} weeks, Valid: {len(valid_agg)} weeks")

Train: 104 weeks, Valid: 39 weeks


## 2. Run: Prophet_Baseline

Fits Prophet with default settings (no tuning of changepoint or seasonality parameters),plus the holiday effects defined above. 

In [3]:
with mlflow.start_run(run_name="Prophet_Baseline") as run:
    model = Prophet(holidays=holidays_df, yearly_seasonality=True, weekly_seasonality=False)
    model.fit(prophet_train)

    future = model.make_future_dataframe(periods=n_valid, freq="W")
    forecast = model.predict(future)

    valid_forecast = forecast.tail(n_valid)["yhat"].values
    score = wmae(valid_agg["Weekly_Sales"].values, valid_forecast, valid_agg["IsHoliday"].values)

    mlflow.log_param("yearly_seasonality", True)
    mlflow.log_param("weekly_seasonality", False)
    mlflow.log_param("series_level", "aggregated_all_stores_depts")
    mlflow.log_metric("valid_wmae", score)

    mlflow.prophet.log_model(
        model,
        artifact_path="prophet_pipeline",
        registered_model_name="Walmart_Prophet_Pipeline",
    )

    print("Prophet aggregated-series WMAE:", score)

13:35:59 - cmdstanpy - INFO - Chain [1] start processing
13:36:00 - cmdstanpy - INFO - Chain [1] done processing
2026/07/12 13:36:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 13:36:05 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'Walmart_Prophet_Pipeline'.
2026/07/12 13:36:08 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Walmart_Prophet_Pipeline, version 1
Created version '1' of model 'Walmart_Prophet_Pipeline'.


Prophet aggregated-series WMAE: 1728319.6579730187
🏃 View run Prophet_Baseline at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/5/runs/16c387bc40ca43e39b5cb635b768bfff
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/5


### Verifying the Logged Model

Loads the model back from the Model Registry and checks it forecasts correctly

In [4]:
loaded = mlflow.prophet.load_model("models:/Walmart_Prophet_Pipeline/1")
check_future = loaded.make_future_dataframe(periods=n_valid, freq="W")
check_forecast = loaded.predict(check_future)
print(check_forecast[["ds", "yhat"]].tail())

            ds          yhat
138 2012-09-23  4.215603e+07
139 2012-09-30  4.344991e+07
140 2012-10-07  4.476451e+07
141 2012-10-14  4.469192e+07
142 2012-10-21  4.382246e+07
